# **County Rainfall and Planting Advisory**

**Analyst:** Amanda Ireri | **Cohort:** 1 Million Devs

An NGO sends SMS planting advice to farmers in five counties. This notebook cleans a hand-typed Machakos rainfall record, analyses all five counties with NumPy, saves a summary CSV and ends with a planting advisory.

In [51]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ", "9", "12", "", "45", "182", "96"]

In [52]:
import csv
import numpy as np

months = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]

## **1: Clean the Machakos data**

**Approach:** The Machakos readings are text, and two are invalid.

`to_int_or_none()` tries to convert one value and catches only `ValueError`. `clean_machakos()` converts every value, averages the valid ones, rounds the average, and uses it to replace each invalid value. The months that were replaced are printed.

In [53]:
def to_int_or_none(value):

    try:
        return int(value.strip())
    except ValueError:
        return None


def clean_machakos(raw_values):

    converted = [to_int_or_none(v) for v in raw_values]
    valid = [v for v in converted if v is not None]
    fill_value = round(sum(valid) / len(valid))

    cleaned = []
    imputed_positions = []
    for position, value in enumerate(converted):
        if value is None:
            cleaned.append(fill_value)
            imputed_positions.append(position)
        else:
            cleaned.append(value)
    return cleaned, imputed_positions, fill_value


machakos_clean, imputed_positions, fill_value = clean_machakos(machakos_raw)
imputed_months = [months[i] for i in imputed_positions]

print("Cleaned Machakos row:", machakos_clean)
print(f"Invalid readings replaced with {fill_value} mm (rounded average of the valid values)")
print("Imputed months:", ", ".join(imputed_months))

Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Invalid readings replaced with 68 mm (rounded average of the valid values)
Imputed months: April, September


## **2: Combine the counties into one NumPy array**

**Approach:** `clean_rows` holds four counties. Adding `[machakos_clean]` to it makes a list of five rows, and `np.array()` turns that into a 5 × 12 table (rows = counties, columns = months).

In [54]:
all_rows = clean_rows + [machakos_clean]
rain = np.array(all_rows)

print("Shape:", rain.shape)
print("Data type:", rain.dtype)

Shape: (5, 12)
Data type: int64


## **3: Annual totals and monthly averages**

**Approach:** `axis=1` collapses across the months, giving one total per county. `axis=0` collapses down the counties, giving one average per month.

In [55]:
annual_totals = rain.sum(axis=1)
monthly_avg = rain.mean(axis=0).round(1)

print("Annual rainfall per county")
for county, total in zip(counties, annual_totals):
    print(f"  {county:<10} {total:>6,} mm")

print("\nAverage rainfall per month (all five counties)")
for month, avg in zip(months, monthly_avg):
    print(f"  {month}  {avg:>6.1f} mm")

Annual rainfall per county
  Nairobi     1,045 mm
  Nakuru        916 mm
  Kisumu      1,371 mm
  Mombasa     1,075 mm
  Machakos      811 mm

Average rainfall per month (all five counties)
  January    47.0 mm
  February    47.6 mm
  March    91.0 mm
  April   164.4 mm
  May   171.2 mm
  June    67.6 mm
  July    56.6 mm
  August    58.2 mm
  September    64.2 mm
  October    67.8 mm
  November   127.4 mm
  December    80.6 mm


## **4: Wettest month per county**

**Approach:** `argmax(axis=1)` returns the position of the largest value in each row. That position is a number, so I look up the month name in the `months` list, and `zip()` pairs each county with its result.

In [56]:
wettest_idx = rain.argmax(axis=1)
wettest_mm = rain.max(axis=1)
wettest_months = [months[i] for i in wettest_idx]

print("Wettest month per county")
for county, month, mm in zip(counties, wettest_months, wettest_mm):
    print(f"  {county:<10} {month}  ({mm} mm)")

Wettest month per county
  Nairobi    April  (219 mm)
  Nakuru     April  (145 mm)
  Kisumu     April  (240 mm)
  Mombasa    May  (240 mm)
  Machakos   November  (182 mm)


## **5: Dry months below 50 mm**

**Approach:** `rain < 50` makes a True/False table of the same shape. `True` counts as 1, so `.sum(axis=1)` counts the dry months in each county.

In [57]:
dry_mask = rain < 50
dry_counts = dry_mask.sum(axis=1)

print("Dry months below 50 mm")
for county, n in zip(counties, dry_counts):
    print(f"  {county:<10} {n}")

Dry months below 50 mm
  Nairobi    4
  Nakuru     3
  Kisumu     0
  Mombasa    2
  Machakos   6


## **6: Planting window**

**Approach:** `rain[:, :-1]` is every month except December and `rain[:, 1:]` is every month except January. Adding them lines up each month with the next one, so each value in `pair_totals` is the rainfall for two consecutive months (Jan+Feb, Feb+Mar, and so on). `argmax` finds the best pair, and the window is named from its start month and the month after.

In [58]:
pair_totals = rain[:, :-1] + rain[:, 1:]
best_start = pair_totals.argmax(axis=1)
best_mm = pair_totals.max(axis=1)
planting_windows = [months[i] + "-" + months[i + 1] for i in best_start]

print("pair_totals shape:", pair_totals.shape)
print("\nRecommended planting window")
for county, window, mm in zip(counties, planting_windows, best_mm):
    print(f"  {county:<10} {window}  ({mm} mm)")

pair_totals shape: (5, 11)

Recommended planting window
  Nairobi    April-May  (395 mm)
  Nakuru     April-May  (265 mm)
  Kisumu     April-May  (420 mm)
  Mombasa    April-May  (390 mm)
  Machakos   November-December  (278 mm)


## **7: Save the results to rainfall_summary.csv**

**Approach:** Each county becomes one dictionary with the five required keys. `csv.DictWriter` writes the header row and then one line per county.

## **Planting Advisory**

Plant in April to May in Nairobi, Nakuru, Kisumu and Mombasa, because this two-month stretch has the most rain in all four counties (395, 265, 420 and 390 mm). Machakos is different: its best window is November to December with 278 mm, so SMS advice there should point farmers to the end-of-year rains. Machakos is also the driest county, with 811 mm a year and 6 dry months, so farmers there should favour drought-tolerant crops and conserve water. Kisumu is the most reliable county, with 1,371 mm a year and no dry months, so planting there carries the lowest drought risk. Caveat: two Machakos readings (April and September) were missing and replaced with the average of 68 mm, so the Machakos window would change only if the true April rainfall was above 138 mm. These figures cover one year, so compare them with long-term averages before sending final advice.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## **Data Quality Notes**

- The Machakos row was typed by hand. Two of its 12 readings were invalid: "N/A" (April) and an empty entry (September).
- Both were replaced with 68 mm, the rounded average of the 10 valid readings (675 ÷ 10 = 67.5, rounded to 68).
- One valid reading (" 20 ") had extra spaces; it was cleaned and kept as 20.
- The two imputed values are estimates, not measurements. They are 2 of the 60 readings (3.3%).
- The Machakos planting window stays November to December unless the true April reading was above 138 mm.
- The Machakos dry-month count (6) could be 7 if September's true rainfall was below 50 mm.
- The data covers only one year, and the other four counties' readings were not checked beyond what was supplied.